# word 2 vec!

My personal implementation of mapping words into vector spaces.

Inspired by the original word2vec paper : https://arxiv.org/abs/1301.3781

Inspired by the follow-up to the original paper : https://proceedings.neurips.cc/paper_files/paper/2013/file/9aa42b31882ec039965f3c4923ce901b-Paper.pdf

### Setup

In [ ]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 74.8 MB/s eta 0:00:00


In [ ]:
# Imports
from datasets import load_dataset

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from collections import Counter

import pytorch_lightning as pl
from pytorch_lightning import Trainer

In [ ]:
# Hyperparams

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = 10000
WINDOW_SIZE = 5
SEED = 42

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-v1/test-00000-of-00001.parque(…):   0%|          | 0.00/685k [00:00<?, ?B/s]

wikitext-2-v1/train-00000-of-00001.parqu(…):   0%|          | 0.00/6.07M [00:00<?, ?B/s]

wikitext-2-v1/validation-00000-of-00001.(…):   0%|          | 0.00/618k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## 1 - Preprocessing

First, we convert the text into a set of *tokens*, to have a standard vocabulary size. For our tokens, we just use the lowercase of every word, although modern LLMs further segment these words.

In [ ]:
def tokenize(batch):
  return {"tokens": [text.lower().split() for text in batch["text"]]}

In [ ]:
ds_tokenized = ds['train'].map(tokenize, batched=True)

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

We flatten the tokens into a one-dimensional list.

In [ ]:
all_tokens = [token for example in ds_tokenized for token in example["tokens"]]

We decide to keep the `VOCAB_SIZE` most common words for the sake of training time.

In [ ]:
counter = Counter()
counter.update(all_tokens)
vocab = set(x for (x, y) in counter.most_common(VOCAB_SIZE))

In [ ]:
tokens_filtered = [token for token in all_tokens if token in vocab]

Our neural nets cannot take text as input. Instead, we map each unique token to a unique integer value using a hashmap.

In [ ]:
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

Finally, we drop tokens probabilistically depending on how often they appear, giving rarer words a chance to appear.

We are left with a token `corpus`.

In [ ]:
token_probs = [1 - np.sqrt(1e-5 / freq_token) for freq_token in counter.values()]
token_probs = dict(zip(vocab, token_probs / np.sum(token_probs)))

In [ ]:
corpus = [token for token in tokens_filtered if np.random.rand() > token_probs[token]]
corpus_idx = [word2idx[token] for token in corpus]

### Section II - Pair Generation

We generate a sampling table, following the skipgram architecture. Instead of softmaxxing over the whole vocab, we pick a few random negative samples (words that don't semantically sit near the center word).


In [ ]:
neg_vals = np.power(np.array(list(token_probs.values())), 3/4)
neg_vals = neg_vals / np.sum(neg_vals)
neg_probs = dict(zip(vocab, neg_vals))

In [ ]:
counts = (neg_vals * 1e8).astype(int)
sampling_table = np.repeat(np.arange(0, len(vocab)), counts)
np.random.shuffle(sampling_table)

In [ ]:
def generate_pairs(corpus_idx, word2idx=word2idx, window_size=WINDOW_SIZE):
  for idx, word in enumerate(corpus_idx):
    start = max(0, idx - window_size)
    end = min(len(corpus_idx), idx + window_size + 1)

    for context_idx in corpus_idx[start:end]:
      if context_idx != word:
        yield (word2idx[word], word2idx[context_idx])

In [ ]:
pairs = np.asarray(list(generate_pairs(corpus)), dtype=np.int64)

### Section 3

In [ ]:
class Word2VecDataset(Dataset):
  def __init__(self, d_vocab, pairs, sampling_table):
    super().__init__()
    self.pairs = pairs
    self.d_vocab = d_vocab
    self.sampling_table = sampling_table

  def __len__(self):
    return len(pairs)

  def __getitem__(self, idx):
    center, context = self.pairs[idx]
    neg_items = torch.tensor(self.sampling_table[np.random.randint(len(self.sampling_table), size=5)])

    return (center, context, neg_items)


In [ ]:
# Set and Loader

word2vec_train = Word2VecDataset(VOCAB_SIZE, pairs, sampling_table)
train_loader = DataLoader(word2vec_train, batch_size=512, shuffle=True, num_workers=7 if torch.cuda.is_available() else 0)

### Section 4

In [ ]:
class word2vec(nn.Module):
  def __init__(self, d_vocab, embed_dim=512):
    super().__init__()
    self.d_vocab = d_vocab
    self.embed_dim = embed_dim
    self.in_embed = nn.Embedding(d_vocab, embed_dim)
    self.out_embed = nn.Embedding(d_vocab, embed_dim)

  def forward(self, center, context, negatives):

    center_embed = self.in_embed(center) # shape is [batch, embed_dim]
    context_embed = self.out_embed(context) # shape is [batch, embed_dim]
    neg_embed = self.out_embed(negatives) # shape is [batch, num_negs, embed_dim,]

    pos_score = torch.sum(torch.mul(center_embed, context_embed), dim=1)
    neg_score = torch.bmm(neg_embed, center_embed.unsqueeze(2)).squeeze(-1) # [B, E, 1], [B, K, E] --> [B, K, 1] (squeeze) --> [B, K]

    loss = F.logsigmoid(pos_score) + F.logsigmoid(-neg_score).sum(dim=1)

    return -(loss.mean())

In [ ]:
class lightning_word2vec(pl.LightningModule):
  def __init__(self, model : nn.Module):
    super().__init__()
    self.model = model

  def training_step(self, batch, batch_idx):
    center, context, neg_pairs = batch
    loss = self.model(center, context, neg_pairs)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.model.parameters(), lr=0.025)

### Trainer

In [ ]:
# Instantiate the net and the wrapper.

net = word2vec(VOCAB_SIZE).to(DEVICE)
net_wrap = lightning_word2vec(net)

In [ ]:
# Instantiate the net and the wrapper.

trainer = Trainer(
    max_epochs=5,
    accelerator='cuda' if torch.cuda.is_available() else 'cpu',
    devices=1
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
# Train the model.

trainer.fit(
    model=net_wrap,
    train_dataloaders=train_loader
)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ word2vec │ 10.2 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 10.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.2 M                                                                                               
Total estimated model params size (MB): 40                                                                         
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

FileNotFoundError: [Errno 2] No such file or directory

In [ ]:
net.parameters()

In [ ]:
w = net.in_embed.weight

# fin.